In [1]:
# -*- coding: utf-8 -*-
"""
Detect script files that are called from CI YAML files and list them in a CSV.

What this does:
- Scans all *.yml / *.yaml files in All_Config_Files.
- Looks for calls to script files with typical script extensions:
  .sh, .bash, .zsh, .ksh, .bat, .cmd, .ps1, .psm1, .psd1
- Normalizes the matched script paths and outputs one row per unique script
  per (repo, YAML file).

Output columns:
- full_name      : owner.repo (derived from flattened filename)
- yml_filename   : the flattened YAML filename
- ci_platform    : provider token from filename (github_actions, gitlab, etc.), or ""
- script_path    : path as it appears in the YAML (normalized slashes)
- script_basename: last path component (e.g., run_tests.sh)

This does NOT open or analyze script content yet; it's just the index of calls.
"""

import re
import os
import pandas as pd
from pathlib import Path
from typing import List, Iterable

# === CONFIG ===
CONFIG_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\All_Config_Files")
OUTPUT_CSV = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\Script_Files_Review\Scripts_Called_From_YML.csv")

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# --- Helpers copied from your CI-analysis conventions ---

def extract_full_name_from_file(filename: str) -> str:
    """
    owner.repo from flattened filename: owner.repo__ci_platform++file.yml
    If pattern is missing, fall back to stem.
    """
    fname = filename.lower()
    if "__" in fname:
        return fname.split("__", 1)[0]
    return Path(fname).stem

def extract_ci_platform(filename: str) -> str:
    """
    ci_platform from flattened filename: owner.repo__ci_platform++file.yml
    If pattern is missing, return "".
    """
    fname = filename.lower()
    if "__" in fname and "++" in fname:
        return fname.split("__", 1)[1].split("++", 1)[0]
    return ""

# Strip full-line comments (#, //, REM, ::) so we don't pick up scripts in comments
COMMENT_LINE_RE = re.compile(r'(?m)^\s*(#|//|REM\b|::).*?$')

def strip_comments(raw: str) -> str:
    return COMMENT_LINE_RE.sub("", raw or "")

# --- Script detection ---

# Script extensions we care about
SCRIPT_EXTS = (
    "sh", "bash", "zsh", "ksh",
    "bat", "cmd",
    "ps1", "psm1", "psd1",
)

# Regex to find script-like paths in the YAML content.
# Examples matched:
#   ./ci/run_android_tests.sh
#   .\scripts\run.bat
#   scripts/run_tests.ps1
#   bash ./ci/run.sh
#   sh scripts/run.sh
SCRIPT_PATH_RE = re.compile(
    rf'(?mi)(?:^|\s)(?P<prefix>\.\/|\.\\|/)?(?P<path>[\w\-.\/\\]+?\.(?:{"|".join(SCRIPT_EXTS)}))\b'
)

def find_scripts_in_text(text: str) -> List[str]:
    """
    Return a list of normalized script paths detected in the text.
    Normalization: convert backslashes to forward slashes, keep ./ or / prefix if present.
    """
    scripts = []
    seen = set()

    for m in SCRIPT_PATH_RE.finditer(text):
        prefix = m.group("prefix") or ""
        path = m.group("path") or ""
        token = (prefix + path).replace("\\", "/")
        token = token.strip()

        if not token:
            continue
        if token in seen:
            continue
        seen.add(token)
        scripts.append(token)

    return scripts

# --- Main ---

def main():
    rows = []

    # We only care about YAML files in this phase
    for f in sorted(CONFIG_DIR.iterdir()):
        if not f.is_file():
            continue
        if f.suffix.lower() not in (".yml", ".yaml"):
            continue

        filename = f.name
        full_name = extract_full_name_from_file(filename)
        ci_platform = extract_ci_platform(filename)

        try:
            raw = f.read_text(encoding="utf-8", errors="ignore")
        except Exception:
            raw = ""

        text = strip_comments(raw)
        script_paths = find_scripts_in_text(text)

        for sp in script_paths:
            # last path component after / (already normalized)
            basename = sp.split("/")[-1] if "/" in sp else sp
            rows.append({
                "full_name": full_name,
                "yml_filename": filename,
                "ci_platform": ci_platform,
                "script_path": sp,
                "script_basename": basename,
            })

    df = pd.DataFrame(rows)
    if not df.empty:
        df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    else:
        # Still create an empty file with header so downstream code doesn't break
        df = pd.DataFrame(columns=["full_name", "yml_filename", "ci_platform", "script_path", "script_basename"])
        df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    print(f"Saved script-call index: {OUTPUT_CSV} (rows={len(df)})")

if __name__ == "__main__":
    main()


Saved script-call index: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\Script_Files_Review\Scripts_Called_From_YML.csv (rows=3464)


In [3]:
# -*- coding: utf-8 -*-
"""
Download all scripts referenced from CI YAML files (Scripts_Called_From_YML.csv)
by:
  1) Mapping each full_name (owner.repo) to its GitHub URL via URL_List.csv
  2) Shallow-cloning only those repos
  3) Copying the requested script files into All_Called_Scripts, preserving
     repo-relative paths where possible.

After running this, you can point your existing script-instrumentation scanner
at:
  C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\Script_Files_Review\All_Called_Scripts
and reuse the same detection logic you already have.
"""

from __future__ import annotations

import os
import shutil
import subprocess
from pathlib import Path
from typing import Dict, Set, List
from urllib.parse import urlparse

import pandas as pd

# ========= CONFIG =========
BASE_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1")

URL_LIST_CSV = BASE_DIR / "URL_List.csv"

SCRIPTS_REVIEW_DIR = BASE_DIR / "Script_Files_Review"
SCRIPTS_CALLED_CSV = SCRIPTS_REVIEW_DIR / "Scripts_Called_From_YML.csv"

CLONE_ROOT = SCRIPTS_REVIEW_DIR / "Script_Clones"        # temporary clones
SCRIPTS_OUT_DIR = SCRIPTS_REVIEW_DIR / "All_Called_Scripts"  # final scripts

INDEX_CSV = SCRIPTS_REVIEW_DIR / "Downloaded_Scripts_Index.csv"

for p in [CLONE_ROOT, SCRIPTS_OUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)


def run(cmd: List[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess[str]:
    """Small wrapper around subprocess.run with text output."""
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=check,
    )


def build_full_name_mapping(url_list_csv: Path) -> Dict[str, str]:
    """
    Read URL_List.csv and build a mapping:
        full_name (owner.repo, lower) -> github_url
    """
    df = pd.read_csv(url_list_csv)
    df.columns = [c.strip().lower() for c in df.columns]

    if "github_url" not in df.columns:
        raise RuntimeError("URL_List.csv must contain a 'github_url' column.")

    df = df[df["github_url"].notna()]
    df["github_url"] = df["github_url"].astype(str).str.strip()

    mapping: Dict[str, str] = {}
    for url in df["github_url"]:
        if not url.startswith("http"):
            continue
        parsed = urlparse(url)
        parts = parsed.path.strip("/").split("/")
        if len(parts) < 2:
            continue
        owner = parts[0]
        repo = parts[1].replace(".git", "")
        full_name = f"{owner}.{repo}".lower()
        mapping[full_name] = url
    return mapping


def load_scripts_by_repo(scripts_csv: Path) -> Dict[str, Set[tuple[str, str]]]:
    """
    Read Scripts_Called_From_YML.csv and group script requests by repo.

    Returns:
        { full_name (lower): set[(script_path, script_basename), ...] }
    """
    df = pd.read_csv(scripts_csv)
    required_cols = {"full_name", "script_path", "script_basename"}
    if not required_cols.issubset(set(df.columns)):
        raise RuntimeError(
            f"Scripts_Called_From_YML.csv must contain columns: {sorted(required_cols)}"
        )

    scripts_by_repo: Dict[str, Set[tuple[str, str]]] = {}

    for _, row in df.iterrows():
        full_name = str(row["full_name"]).strip().lower()
        script_path = str(row["script_path"]).strip()
        script_base = str(row["script_basename"]).strip()

        if not full_name or not script_path or not script_base:
            continue

        # Skip obviously non-local things (URLs, etc.)
        if script_path.startswith("http://") or script_path.startswith("https://"):
            continue

        scripts_by_repo.setdefault(full_name, set()).add((script_path, script_base))

    return scripts_by_repo


def detect_default_branch(repo_url: str) -> str:
    """
    Use 'git ls-remote --symref <url> HEAD' to detect default branch, with
    fallback to main/master.
    """
    try:
        out = run(["git", "ls-remote", "--symref", repo_url, "HEAD"]).stdout
        for line in out.splitlines():
            s = line.strip()
            if s.startswith("ref: ") and s.endswith("HEAD"):
                ref = s.split()[1]
                if ref.startswith("refs/heads/"):
                    return ref.split("/", 2)[2]
    except Exception:
        pass

    # Fallback guesses
    for guess in ("main", "master"):
        try:
            run(["git", "ls-remote", repo_url, f"refs/heads/{guess}"], check=True)
            return guess
        except Exception:
            continue

    raise RuntimeError("Could not determine default branch for: " + repo_url)


def clone_repo_if_needed(repo_url: str, full_name: str) -> Path:
    """
    Shallow-clone the default branch for this repo under CLONE_ROOT/<full_name>.
    If already exists, skip cloning and reuse.
    """
    repo_dir = CLONE_ROOT / full_name.replace("/", "__")
    if repo_dir.exists():
        print(f"[INFO] Reusing existing clone for {full_name}: {repo_dir}")
        return repo_dir

    print(f"\n=== Cloning {full_name} ===")
    default_branch = detect_default_branch(repo_url)
    print(f"[INFO] Default branch for {full_name}: {default_branch}")

    try:
        run([
            "git", "clone",
            "--depth", "1",
            "--single-branch",
            "--branch", default_branch,
            repo_url,
            str(repo_dir),
        ])
        print(f"[OK] Clone complete: {repo_dir}")
    except subprocess.CalledProcessError as e:
        print(f"[ERROR] Clone failed for {full_name} ({repo_url})")
        print(e.stdout)
        raise

    return repo_dir


def normalize_script_relpath(script_path: str) -> str:
    """
    Clean up script paths from YAML, e.g. './scripts/run.sh' -> 'scripts/run.sh'.
    """
    p = script_path.strip()
    # Strip leading ./ or .\
    while p.startswith("./") or p.startswith(".\\"):
        p = p[2:]
    p = p.replace("\\", "/")
    return p


def copy_script_if_exists(
    repo_dir: Path,
    full_name: str,
    script_path: str,
    script_basename: str,
) -> tuple[bool, Path | None, str]:
    """
    Try to copy the requested script into SCRIPTS_OUT_DIR/full_name/<relpath>.

    Returns:
        (found: bool, dest_path: Path | None, found_mode: str)
        where found_mode is "exact", "basename_fallback", or "".
    """
    rel = normalize_script_relpath(script_path)
    candidate = repo_dir / rel

    # 1) Exact path
    if candidate.is_file():
        dest = SCRIPTS_OUT_DIR / full_name / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(candidate, dest)
        return True, dest, "exact"

    # 2) Fallback: search by basename anywhere in repo
    matches = list(repo_dir.rglob(script_basename))
    if matches:
        # If multiple, pick the shortest relative path (closest to root)
        matches.sort(key=lambda p: len(p.relative_to(repo_dir).parts))
        best = matches[0]
        rel_best = best.relative_to(repo_dir)
        dest = SCRIPTS_OUT_DIR / full_name / rel_best
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(best, dest)
        return True, dest, "basename_fallback"

    # Not found
    return False, None, ""


def main():
    # 1) Build mapping full_name -> github_url
    full_name_to_url = build_full_name_mapping(URL_LIST_CSV)
    print(f"[INFO] Loaded {len(full_name_to_url)} repos from URL_List.csv")

    # 2) Load scripts grouped by repo
    scripts_by_repo = load_scripts_by_repo(SCRIPTS_CALLED_CSV)
    print(f"[INFO] Found script references for {len(scripts_by_repo)} repos")

    index_rows = []

    for full_name, scripts in sorted(scripts_by_repo.items()):
        repo_url = full_name_to_url.get(full_name)
        if not repo_url:
            print(f"[WARN] No GitHub URL found for {full_name} in URL_List.csv; skipping.")
            continue

        try:
            repo_dir = clone_repo_if_needed(repo_url, full_name)
        except Exception:
            print(f"[ERROR] Skipping repo due to clone failure: {full_name}")
            continue

        for script_path, script_basename in sorted(scripts):
            found, dest, mode = copy_script_if_exists(
                repo_dir, full_name, script_path, script_basename
            )
            if not found:
                print(f"[WARN] Script not found in repo {full_name}: {script_path} ({script_basename})")
                index_rows.append({
                    "full_name": full_name,
                    "repo_url": repo_url,
                    "script_path": script_path,
                    "script_basename": script_basename,
                    "found": False,
                    "found_mode": "",
                    "saved_path": "",
                })
            else:
                print(f"[OK] Saved script for {full_name}: {script_path} -> {dest} [{mode}]")
                index_rows.append({
                    "full_name": full_name,
                    "repo_url": repo_url,
                    "script_path": script_path,
                    "script_basename": script_basename,
                    "found": True,
                    "found_mode": mode,
                    "saved_path": str(dest),
                })

    if index_rows:
        df_idx = pd.DataFrame(index_rows)
        df_idx.to_csv(INDEX_CSV, index=False, encoding="utf-8-sig")
        print(f"\n[INFO] Wrote index of downloaded scripts to: {INDEX_CSV}")
    else:
        print("\n[INFO] No scripts were downloaded (nothing to index).")


if __name__ == "__main__":
    main()


[INFO] Loaded 4697 repos from URL_List.csv
[INFO] Found script references for 743 repos

=== Cloning 1q23lyc45.kitsunemagisk ===
[INFO] Default branch for 1q23lyc45.kitsunemagisk: kitsune
[OK] Clone complete: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\Script_Files_Review\Script_Clones\1q23lyc45.kitsunemagisk
[OK] Saved script for 1q23lyc45.kitsunemagisk: scripts/avd_test.sh -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\Script_Files_Review\All_Called_Scripts\1q23lyc45.kitsunemagisk\scripts\avd_test.sh [exact]

=== Cloning 2003scape.rsc-c ===
[INFO] Default branch for 2003scape.rsc-c: master
[OK] Clone complete: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\Script_Files_Review\Script_Clones\2003scape.rsc-c
[OK] Saved script for 2003scape.rsc-c: ./autogen.sh -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\Script_Files_Review\All_Called_Scripts\2003scape.rsc-c\cglm\autogen.sh [basename_fallback]
[OK] Saved script for 2003scape.rsc-c: ./bui

In [ ]:
# read the called scripts for isntru test signals

In [2]:
# -*- coding: utf-8 -*-
"""
Scan script files (that are called from CI YAML files) for instrumentation-testing signals and output:
filename, full_name, ci_platform, instru_t_ci_signal, confidence, confidence_reason,
execution_environment, test_invocation, flutter_integ_t_signal, flutter_integ_t_d,
third_party_env_label, called_from_ymls, script_paths

Notes:
- Uses the SAME detection logic as the YAML V6.0 scanner (Gradle / ADB / 3P labs / Flutter IT).
- Only scripts that are actually referenced in YAML (as recorded in Scripts_Called_From_YML.csv) are analyzed.
- One row per script file found in All_Config_Files.
"""

import os
import re
import pandas as pd
from pathlib import Path
from typing import List, Pattern, Tuple, Dict, Any, Iterable

# === CONFIG ===
CONFIG_DIR = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\All_Config_Files")

SCRIPTS_META_CSV = Path(
    r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\Script_Files_Review\Scripts_Called_From_YML.csv"
)
OUTPUT_CSV = Path(
    r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\Script_Files_Review\Script_Files_Instru.csv"
)

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# We treat these as "script" files inside All_Config_Files.
SCRIPT_EXTS = {
    ".sh", ".bash", ".zsh", ".ksh",
    ".bat", ".cmd",
    ".ps1", ".psm1", ".psd1",
}

# === Helpers ===
def extract_full_name_from_file(filename: str) -> str:
    fname = filename.lower()
    if "__" in fname:
        return fname.split("__", 1)[0]
    return Path(fname).stem

def extract_ci_platform(filename: str) -> str:
    fname = filename.lower()
    if "__" in fname and "++" in fname:
        return fname.split("__", 1)[1].split("++", 1)[0]
    return ""

def compile_any(patterns: List[str], flags=re.I | re.M) -> List[Pattern]:
    return [re.compile(p, flags) for p in patterns]

def any_match(patterns: Iterable[Pattern], text: str) -> bool:
    return any(p.search(text) for p in patterns)

def unique_preserve(seq: Iterable[str]) -> List[str]:
    seen, out = set(), []
    for x in seq:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def collect_hits_with_groups(patterns: List[Tuple[str, str, List[Pattern]]], text: str):
    labels, groups = [], []
    for grp, lbl, pats in patterns:
        if any_match(pats, text):
            labels.append(lbl)
            groups.append(grp)
    return unique_preserve(labels), unique_preserve(groups)

# Strip comments (same as YAML scanner)
COMMENT_LINE_RE = re.compile(r'(?m)^\s*(#|//|REM\b|::).*?$')
def strip_comments(raw: str) -> str:
    return COMMENT_LINE_RE.sub("", raw or "")

# Gradle + shell prefixes (same as YAML scanner)
GRADLE_PREFIX = (
    r'^\s*'
    r'(?:\S+=\S+\s+)*'
    r'(?:sudo\s+)?'
    r'(?:(?:bash|sh)\s+-c[l]?\s+[\'"]?)?'
    r'(?:[^#\n;]*?&&\s+)?'
    r'(?:cd\s+\S+\s+&&\s+)?'
    r'(?:\./|\.\\)?gradle(?:w)?(?:\.bat)?'
)
GRADLE_ANYWHERE_RE = re.compile(r'(?i)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*')
GRADLE_BUILD_ACTION_RE = re.compile(r'(?mi)\buses\s*:\s*(gradle/gradle-build-action|gradle/actions/setup-gradle)@')

NON_TEST_PREFIX = (
    r'(?:assemble|bundle|package|compile|merge|process|generate|install|uninstall|jacoco|lint|publish|sign|upload)'
)
SHELL_PREFIX = (
    r'(?:\S+=\S+\s+)*(?:sudo\s+)?'
    r'(?:(?:bash|sh|pwsh|powershell)\s+-c\s+[\'"]?)?'
    r'(?:[^#\n;]*?&&\s+)?'
)

# Emulator/ADB lines
EMULATOR_LINE = rf'(?mi)^[^\n]*{SHELL_PREFIX}(?:\s|^)(?:(?:\./|\.\\)?(?:emulator)(?:\.exe)?)\b[^\n]*-avd\s+\S+'
ADB_WAIT_LINE = rf'(?mi)^[^\n]*{SHELL_PREFIX}adb\s+wait[- ]?for[- ]?device\b'
ADB_SERIAL_LINE = rf'(?mi)^[^\n]*{SHELL_PREFIX}adb\s+-s\s+(?:emulator-\d+|localhost:\d+|127\.0\.0\.1:\d+)'

# Device/Environment sources (NOT triggers) – unchanged from YAML scanner
DEVICE_SOURCES = [
    ("Real_Device", "adb -s <serial> (physical)", [
        r'(?m)^\s*adb\s+-s\s+(?!emulator-\d+\b)(?!localhost:\d+\b)(?!127\.0\.0\.1:\d+\b)\S+\b'
    ]),
    ("Emulator", "adb -s emulator-serial", [ADB_SERIAL_LINE]),
    ("Emulator", "adb wait-for-device",   [ADB_WAIT_LINE]),
    ("Emulator", "emulator -avd/@",       [EMULATOR_LINE]),
    ("Emulator", "android-wait-for-emulator", [r'(?m)^\s*(?:\./)?android-wait-for-emulator\b']),
    ("Emulator", "start-emulator.sh", [r'(?m)^\s*start-emulator\.sh\b']),
    ("Emulator", "android create avd", [r'\bandroid\b[^\n]*\bcreate\s+avd\b']),
    ("Emulator", "circleci android orb", [
        r"(?mi)^\s*(?:-\s*)?android/(?:start-emulator-and-run-tests|create-avd|launch-emulator)\s*:",
        r"(?mi)^\s*system-image\s*:\s*system-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*);(?:default|google_apis)[^\s]*"
    ]),
    ("Emulator", "reactivecircus runner", [
        r'(?mi)\buses\s*:\s*reactivecircus/android-emulator-runner@[\w\.\-]+'
    ]),
    ("Emulator", "malinskiy runner", [
        r'(?mi)\buses\s*:\s*malinskiy/action-android/emulator-run-cmd@[\w\.\-]+'
    ]),
    ("Emulator", "sys-img component", [
        r'(?m)^\s*-\s*sys-img-[^\s]*-android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b',
        r'(?m)^\s*-\s*sys-img-[^\s]*-google_apis-[^\s]*(?:\d+|\$[A-Z_][A-Z0-9_]*)\b'
    ]),
    ("Emulator", "avdmanager", [r'(?m)^\s*\S*avdmanager\b']),
    ("Emulator", "sdkmanager system-images/emulator", [
        r'^\s*\S*sdkmanager\b[^\n"]*"system-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)[^"\n]*"',
        r'^\s*\S*sdkmanager\b[^\n]*\bsystem-images;android-(?:\d+|\$[A-Z_][A-Z0-9_]*)\b',
    ]),
    # Weak hints (filtered if no strong signals)
    ("Emulator", "headless flag", [
        r'(?mi)\b-no-?audio\b', r'(?mi)\b-no-window\b', r'(?mi)\b-no-boot-anim\b'
    ]),
    ("Emulator", "avd-name", [r'(?mi)^\s*avd[-_ ]?name\s*:\s*\S+']),
    ("Emulator", "api-level", [r'(?mi)\bapi[-_ ]?level\s*:\s*\d{2,}|\bapi_level\s*:\s*\d{2,}']),
    ("Emulator", "abi/arch", [r'\b(abi|arch)\b\s*:?\s*(x86|x86_64|arm64|armeabi)']),
    ("Emulator", "target image", [r'\btarget\s*:\s*(google_apis|google_apis_playstore|aosp.*)']),
    ("Emulator", "device name", [r'\b(avd[-_ ]?name|device)\b\s*:\s*pixel']),
    # Third-party labs (environment)
    ("Third_Party_Lab", "gcloud firebase", [
        r'(?mi)^[^\n]*\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run\b'
    ]),
    ("Third_Party_Lab", "saucectl",        [r'(?mi)^[^\n]*\bsaucectl(?:\s+run)?\b']),
    ("Third_Party_Lab", "browserstack/bstack", [r'(?i)\b(browserstack|bstack)\b']),
    ("Third_Party_Lab", "appcenter test",  [
        r'(?mi)^[^\n]*\bappcenter\s+test\s+run\s+android\b'
    ]),
    ("Third_Party_Lab", "maestro cloud",   [r'(?mi)^[^\n]*\bmaestro\s+cloud\b']),
    ("Third_Party_Lab", "emulator.wtf action", [
        r'(?mi)^\s*uses\s*:\s*emulator-wtf/run-tests@[\w\.\-]+',
        r'(?i)\bemulator\.wtf\b'
    ]),
    # Other emulator actions
    ("Emulator", "other gha emulator", [
        r'(?mi)^\s*uses\s*:\s*vgaidarji/android-github-actions-emulator@[\w\.\-]+',
        r'(?mi)^\s*uses\s*:\s*(?!reactivecircus/android-emulator-runner@)'
        r'(?!malinskiy/action-android/emulator-run-cmd@)'
        r'(?!emulator-wtf/run-tests@)'
        r'[\w\.-]+/[\w\./-]*android[\w\./-]*(?:\bemulator\b|\bavd\b)[\w\./-]*@[\w\.\-]+'
    ]),
]
DEVICE_PATTERNS = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in DEVICE_SOURCES]

# Sanitizers
EXCLUDED_TASK_SEGMENT_RE = re.compile(
    r'(^|\s)(?:-x|--exclude-task)\s+(["\']?)[:\w\.-]*(?:androidtest|baselineprofile)[\w:\.-]*\2\b',
    re.IGNORECASE | re.MULTILINE,
)
GHA_EXPR_RE = re.compile(r"\${{\s*[^}]+}}")

def remove_excluded_gradle_tasks(text: str) -> str:
    return EXCLUDED_TASK_SEGMENT_RE.sub(lambda m: (m.group(1) or " "), text or "")

def pre_sanitize(text: str) -> str:
    t = remove_excluded_gradle_tasks(text)
    return GHA_EXPR_RE.sub("", t or "")

# Triggers (primary + anywhere) — same as YAML scanner (includes BaselineProfile & connectedBenchmarkAndroidTest)
TRIGGER_SOURCES_PRIMARY = [
    ("Gradle",  "connectedAndroidTest",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connectedandroidtest\b[^\n\r]*']),
    ("Gradle", "connected.*Android.*", [
        rf'''(?mix){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connected[a-z0-9:_-]*android[a-z0-9:_-]*test\b(?![^\n\r]*\b(?:{NON_TEST_PREFIX})[\w-]*androidtest\b)[^\n\r]*'''
    ]),
    ("Gradle",  "connectedCheck",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connectedcheck\b[^\n\r]*']),
    ("Gradle",  "cAT shorthand",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*cAT\b[^\n\r]*']),
    ("Gradle",  "deviceCheck",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*(?:devicecheck|alldevicechecks)\b[^\n\r]*']),
    ("Gradle",  "managedDevice AndroidTest",
     [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?!{NON_TEST_PREFIX})(?:manageddevice|device)[\w:-]*androidtest\b[^\n\r]*']),
    ("Gradle", "variant/device AndroidTest",
     [rf'''(?mix){GRADLE_PREFIX}[^\n\r]*\b(?:(?:[:\w-]+:)*(?!{NON_TEST_PREFIX})(?!connected)(?!spoon)(?!marathon)[A-Za-z0-9][\w-]*androidtest\b)[^\n\r]*''']),
    ("Gradle",  "Spoon",    [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\bspoon(?:\w*androidtest)?\b']),
    ("Gradle",  "Marathon", [rf'(?mi){GRADLE_PREFIX}[^\n\r]*\bmarathon(?:\w*androidtest)?\b']),
    ("ADB",     "am instrument", [r'(?mi)^[^\n]*\bam\s+instrument\b']),
    # 3P CLIs (instr only)
    ("Third_Party_Lab", "gcloud firebase (instr)", [
        r'(?mi)\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run[^\n]*\b(--test\b|--type\s+instrumentation\b)'
    ]),
    ("Third_Party_Lab", "flank",            [r'(?mi)^[^\n]*\bflank\s+android\s+run\b']),
    ("Third_Party_Lab", "saucectl",         [r'(?mi)^[^\n]*\bsaucectl(?:\s+run)?\b']),
    ("Third_Party_Lab", "appcenter run",    [
        r'(?mi)^[^\n]*\bappcenter\s+test\s+run\s+android\b'
    ]),
    ("Third_Party_Lab", "emulator.wtf run", [
        r'(?mi)^\s*uses\s*:\s*emulator-wtf/run-tests@[\w\.\-]+',
        r'(?i)\bemulator\.wtf\b'
    ]),
    # BaselineProfile + connectedBenchmarkAndroidTest
    ("Gradle", "generateBaselineProfile", [
        rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*generate(?:\w*?)baselineprofile\b[^\n\r]*'
    ]),
    ("Gradle", "collectBaselineProfile",  [
        rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*collect(?:\w*?)baselineprofile\b[^\n\r]*'
    ]),
    ("Gradle", "connectedBenchmarkAndroidTest", [
        rf'(?mi){GRADLE_PREFIX}[^\n\r]*\b(?:[:\w-]+:)*connectedbenchmarkandroidtest\b[^\n\r]*'
    ]),
]
TRIGGER_SOURCES_ANYWHERE = [
    ("Gradle", "connected (anywhere)", [
        r'(?mix)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*\b(?:[:\w-]+:)*connected[a-z0-9:_-]*android[a-z0-9:_-]*test\b'
    ]),
    ("Gradle", "connectedAndroidTest (anywhere)", [
        r'(?mi)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*\bconnectedandroidtest\b'
    ]),
    ("Gradle", "connectedCheck (anywhere)", [
        r'(?mi)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*\b(?:[:\w-]+:)*connectedcheck\b'
    ]),
    ("Gradle", "managedDevice AndroidTest (anywhere)", [
        rf'(?mix)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*\b(?:(?:[:\w-]+:)*(?!{NON_TEST_PREFIX})(?!connected)(?!spoon)(?!marathon)[A-Za-z0-9][\w-]*androidtest\b)'
    ]),
    ("Gradle", "variant/device AndroidTest (anywhere)", [
        rf'(?mix)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*\b(?:(?:[:\w-]+:)*(?!{NON_TEST_PREFIX})[\w-]*androidtest\b)'
    ]),
    ("Gradle", "Spoon (anywhere)",   [
        r'(?mi)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*\bspoon(?:\w*androidtest)?\b'
    ]),
    ("Gradle", "Marathon (anywhere)",[
        r'(?mi)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*\bmarathon(?:\w*androidtest)?\b'
    ]),
    # BaselineProfile + connectedBenchmarkAndroidTest
    ("Gradle", "generateBaselineProfile (anywhere)", [
        rf'(?mi)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*\bgenerate(?:\w*?)baselineprofile\b'
    ]),
    ("Gradle", "collectBaselineProfile (anywhere)",  [
        rf'(?mi)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*\bcollect(?:\w*?)baselineprofile\b'
    ]),
    ("Gradle", "connectedBenchmarkAndroidTest (anywhere)", [
        rf'(?mi)[^\n]*\bgradle(?:w)?(?:\.bat)?[^\n]*\bconnectedbenchmarkandroidtest\b'
    ]),
]
TRIGGER_PATTERNS_PRIMARY  = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES_PRIMARY]
TRIGGER_PATTERNS_ANYWHERE = [(grp, lbl, compile_any(pats)) for (grp, lbl, pats) in TRIGGER_SOURCES_ANYWHERE]

# Gradle action inputs (same set as YAML scanner)
GHA_GRADLE_INPUTS = compile_any([
    rf'''(?mix)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*connected(?:\${{\s*[^}}]+\s*}}|[^\n\r])*?android(?:\${{\s*[^}}]+\s*}}|[^\n\r])*?test\b''',
    r'(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*connectedcheck\b',
    r'(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*(?:devicecheck|alldevicechecks)\b',
    rf'''(?mix)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:(?:[:\w-]+:)*(?!{NON_TEST_PREFIX})[\w-]*androidtest\b)''',
    r'(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?:[:\w-]+:)*cAT\b',
])
GHA_GMD_INPUTS = compile_any([
    rf'(?mi)^\s*(arguments|tasks|script|cmd)\s*:\s*[^$`\n]*\b(?!(?:[:\w-]+:)*(?:connected[a-z0-9:._-]*|{NON_TEST_PREFIX})[\w:-]*androidtest\b)(?:[:\w-]+:)*[\w:-]*androidtest\b'
])

# Flutter integration test detectors
FLUTTER_IT_LINE = re.compile(r'(?mi)^\s*flutter\s+(?:test|drive)\b[^\n]*')
FLUTTER_IT_ANDROID_HINT = re.compile(r'(?i)(integration_test|--driver\b|/integration_test/)')
FLUTTER_DEVICE_FLAG_RE = re.compile(r'(?i)\s+-d\s+(?P<dev>"[^"]+"|\'[^\']+\'|\S+)')
FLUTTER_DEVICE_IS_ANDROID = re.compile(r'(?i)\b(android|emulator-\d+|sdk\s+gphone|android\s+sdk\s+built\s+for|pixel)\b')
LINUX_HEADLESS_HINTS_RE = re.compile(r'(?mi)^\s*(xvfb-run|export\s+DISPLAY=|sudo\s+Xvfb)\b')

# Provider/env helpers
PROVIDER_PATTERNS = [
    ("emulator-wtf", compile_any([
        r'(?mi)^\s*uses\s*:\s*emulator-wtf/run-tests@',
        r'(?i)\bemulator\.wtf\b'
    ])),
    ("firebase-test-lab", compile_any([
        r'(?mi)\bgcloud(?:\s+beta)?\s+firebase\s+test\s+android\s+run\b',
        r'(?mi)\bflank\s+android\s+run\b'
    ])),
    ("browserstack", compile_any([
        r'(?i)\bbrowserstack\b',
        r'(?i)\bbstack\b'
    ])),
    ("aws-device-farm", compile_any([
        r'(?mi)\baws\s+devicefarm\b'
    ])),
    ("sauce-labs", compile_any([
        r'(?mi)\bsaucectl(?:\s+run)?\b',
        r'(?mi)\bsauce\s+ctl\b'
    ])),
    ("appcenter", compile_any([
        r'(?mi)\bappcenter\s+test\s+run\s+android\b'
    ])),
    ("maestro-cloud", compile_any([
        r'(?mi)\bmaestro\s+cloud\b'
    ])),
]
INLINE_DEVICE_HINTS = compile_any([
    r'(?mi)^\s*devices\s*:\s*\|',
    r'(?mi)\b--device\b',
    r'(?mi)\bmodel\s*=\s*[^,\s]+',
    r'(?mi)\bversion\s*=\s*\d+',
    r'(?mi)\blocale\s*=\s*[-\w]+',
    r'(?mi)\borientation\s*=\s*(portrait|landscape)',
    r'(?mi)^\s*with-orchestrator\s*:\s*true\b',
    r'(?mi)\b--use-orchestrator\b',
    r'(?mi)\bnum-flaky-test-attempts\s*:\s*\d+\b',
    r'(?mi)\b--num-flaky-test-attempts(?:=|\s+)\d+\b',
])
FTL_HAS_INSTRUMENTATION = re.compile(
    r'(?mi)\bgcloud\s+firebase\s+test\s+android\s+run[^\n]*\b(--test\b|--type\s+instrumentation\b)'
)
FTL_APP_ONLY = re.compile(
    r'(?mi)\bgcloud\s+firebase\s+test\s+android\s+run\b(?![^\n]*\b(--test\b|--type\s+instrumentation\b))'
)
CONFIG_FILE_HINTS = compile_any([
    r'(?mi)\.ewtf\.ya?ml\b',
    r'(?mi)\b(flank\.ya?ml|flank\.android\.ya?ml)\b',
    r'(?mi)\b--config(?:=|\s+)\S+',
    r'(?mi)\b(browserstack\.ya?ml)\b',
    r'(?mi)\b(bs(?:config)?\.ya?ml)\b',
])
ORCHESTRATOR_HINTS = compile_any([
    r'(?mi)^\s*with-orchestrator\s*:\s*true\b',
    r'(?mi)\b--use-orchestrator\b'
])
RETRY_HINTS = compile_any([
    r'(?mi)\bnum-flaky-test-attempts\s*:\s*(\d+)\b',
    r'(?mi)\b--num-flaky-test-attempts(?:=|\s+)(\d+)\b'
])

def detect_provider(text: str) -> str:
    for name, pats in PROVIDER_PATTERNS:
        if any_match(pats, text):
            return name
    return "other"

def detect_inline_env(text: str) -> bool:
    return any_match(INLINE_DEVICE_HINTS, text)

def detect_config_env(text: str) -> bool:
    return any_match(CONFIG_FILE_HINTS, text)

def count_devices(text: str) -> int:
    count = 0
    m = re.search(r'(?mi)^\s*devices\s*:\s*\|\s*([\s\S]+)', text)
    if m:
        block = m.group(1)
        lines = [ln for ln in block.splitlines() if ln.strip()]
        pruned = []
        for ln in lines:
            if re.match(r'^\s*\w[\w-]*\s*:\s*', ln):
                break
            pruned.append(ln)
        count += sum(1 for ln in pruned if re.search(r'\bmodel\s*=', ln))
    count += len(re.findall(r'(?mi)\b--device\b', text))
    return count or 0

def detect_orchestrator(text: str) -> bool:
    return any_match(ORCHESTRATOR_HINTS, text)

def detect_retries(text: str) -> int:
    for pat in RETRY_HINTS:
        m = pat.search(text)
        if m:
            try:
                return int(m.group(1))
            except Exception:
                pass
    return 0

# Strong/weak classification
EMULATOR_STRONG_LABELS = {
    "emulator -avd/@", "android-wait-for-emulator", "start-emulator.sh",
    "circle-android wait-for-boot", "reactivecircus runner", "avdmanager",
    "sdkmanager system-images/emulator", "adb -s emulator-serial", "adb wait-for-device",
    "android create avd", "malinskiy runner", "other gha emulator", "circleci android orb",
}
THIRD_PARTY_STRONG_LABELS = {
    "gcloud firebase", "emulator.wtf action", "saucectl",
    "browserstack/bstack", "appcenter test", "maestro cloud"
}
REAL_DEVICE_STRONG_LABELS = {"adb -s <serial> (physical)"}
REAL_DEVICE_GENERIC_ADB = set()
STRONG_DEVICE_LABELS = EMULATOR_STRONG_LABELS | REAL_DEVICE_STRONG_LABELS | THIRD_PARTY_STRONG_LABELS

def filter_weak_device_hints(labels, groups):
    lbl_set = set(labels)
    if not (lbl_set & STRONG_DEVICE_LABELS):
        labels = [l for l in labels if l in STRONG_DEVICE_LABELS]
        if not labels:
            groups = []
    return labels, groups

def reconcile_emulator_vs_real(labels, groups):
    lbls = set(labels)
    if lbls & EMULATOR_STRONG_LABELS:
        lbls -= REAL_DEVICE_GENERIC_ADB
        labels = [l for l in labels if l in lbls]
        if "Real_Device" in groups:
            has_real_after = bool(set(labels) & REAL_DEVICE_STRONG_LABELS)
            if not has_real_after:
                groups = [g for g in groups if g != "Real_Device"]
    return labels, groups

ANDROID_CONTEXT_RE = re.compile(
    r'(?i)\b(adb|avd|emulator|android\s+sdk|system-images;android-|androidtest|connected(check|androidtest)|gcloud\s+firebase\s+test\s+android\s+run)\b'
)

# === Script-related helpers ===

def canonical_script_basename_from_flat(flat_name: str) -> str:
    """
    From a flattened config filename, recover the *original* script basename
    (handles the ++ci_platform++ pattern and __2, __3 collision suffixes).
    """
    lower = flat_name.lower()
    if "++" in lower:
        tail = lower.split("++", 1)[1]
    else:
        tail = lower
    base = os.path.basename(tail)
    stem, ext = os.path.splitext(base)
    m = re.match(r"(.+?)__\d+$", stem)
    if m:
        stem = m.group(1)
    return stem + ext  # e.g. run_tests.sh

def build_script_index(config_dir: Path) -> Dict[Tuple[str, str], List[Tuple[Path, str]]]:
    """
    Map (full_name, original_script_basename_lower) -> list[(Path, ci_platform)].
    Only includes files with SCRIPT_EXTS.
    """
    index: Dict[Tuple[str, str], List[Tuple[Path, str]]] = {}
    for p in config_dir.iterdir():
        if not p.is_file():
            continue
        if p.suffix.lower() not in SCRIPT_EXTS:
            continue
        full_name = extract_full_name_from_file(p.name)
        ci_platform = extract_ci_platform(p.name)
        orig_base = canonical_script_basename_from_flat(p.name)
        key = (full_name, orig_base.lower())
        index.setdefault(key, []).append((p, ci_platform))
    return index

# === Core script analyzer (single script file) ===

def analyze_script_file(path: Path) -> Dict[str, Any]:
    """
    Apply the same instrumentation CI detection logic as the YAML scanner,
    but treating the whole script as a single "job".
    """
    try:
        raw = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        raw = ""

    content = strip_comments(raw)
    content_for_triggers = pre_sanitize(content)

    file_trigger_labels: List[str] = []
    file_device_labels:  List[str] = []
    file_trigger_groups: List[str] = []
    file_device_groups:  List[str] = []
    file_has_test_trigger = False
    file_has_device_with_group = False
    file_gradle_present_any = False
    file_instru_signal_any = False
    file_flutter_devices: List[str] = []

    # Flutter IT flags
    file_flutter_it_present = False
    file_flutter_it_androidish = False

    # metadata-like placeholders for third-party labeling
    file_env_declared = False
    file_env_location = "none"   # none | inline | config
    file_device_count = 0
    file_orchestrator = False
    file_retries = 0

    # -------------- treat whole script as a single "job" --------------
    prov = detect_provider(content_for_triggers)  # currently unused but kept for parity
    inline_decl = detect_inline_env(content_for_triggers)
    config_decl = detect_config_env(content_for_triggers)
    if inline_decl:
        file_env_declared = True
        file_env_location = "inline"
    elif config_decl:
        file_env_declared = True
        file_env_location = "config"

    file_device_count += count_devices(content_for_triggers)
    if detect_orchestrator(content_for_triggers):
        file_orchestrator = True
    file_retries = max(file_retries, detect_retries(content_for_triggers))

    # device & triggers
    dev_labels, dev_groups = collect_hits_with_groups(DEVICE_PATTERNS, content_for_triggers.lower())
    dev_labels, dev_groups = filter_weak_device_hints(dev_labels, dev_groups)
    dev_labels, dev_groups = reconcile_emulator_vs_real(dev_labels, dev_groups)

    trig_labels, trig_groups = collect_hits_with_groups(TRIGGER_PATTERNS_PRIMARY, content_for_triggers.lower())

    # Guard: only count GHA gradle inputs when Gradle is present (command or action) OR tied to emulator action
    if any_match(GHA_GRADLE_INPUTS, content_for_triggers):
        has_gradle_anywhere = bool(
            GRADLE_ANYWHERE_RE.search(content_for_triggers) or GRADLE_BUILD_ACTION_RE.search(content_for_triggers)
        )
        tied_to_emulator_action = bool(re.search(
            r'(?mi)^\s*uses\s*:\s*(?:reactivecircus/android-emulator-runner|malinskiy/action-android/emulator-run-cmd|hannesa2/action-android/emulator-run-cmd)\@',
            content_for_triggers
        ))
        if has_gradle_anywhere or tied_to_emulator_action:
            trig_labels = unique_preserve(trig_labels + ["gha gradle inputs/script"])
            trig_groups = unique_preserve(trig_groups + ["Gradle"])

    if any_match(GHA_GMD_INPUTS, content_for_triggers):
        trig_labels = unique_preserve(trig_labels + ["variant/device AndroidTest"])
        trig_groups = unique_preserve(trig_groups + ["Gradle"])

    # ---- Flutter integration test detection (Android-gated for instrumentation)
    flutter_it_hits = []
    for m in FLUTTER_IT_LINE.finditer(content_for_triggers):
        line = m.group(0)
        if FLUTTER_IT_ANDROID_HINT.search(line) or FLUTTER_DEVICE_IS_ANDROID.search(line):
            flutter_it_hits.append(line)

    flutter_it_android_targeted = False
    if flutter_it_hits:
        trig_labels = unique_preserve(trig_labels + ["flutter integration test"])
        trig_groups = unique_preserve(trig_groups + ["Flutter"])
        file_flutter_it_present = True

        for line in flutter_it_hits:
            d = FLUTTER_DEVICE_FLAG_RE.search(line)
            if d:
                plat = d.group("dev").strip('"\'').lower()
                if (
                    plat == "android"
                    or plat.startswith("emulator-")
                    or "sdk gphone" in plat
                    or "android sdk built for" in plat
                    or "pixel" in plat
                ):
                    flutter_it_android_targeted = True
                    if "android" not in file_flutter_devices:
                        file_flutter_devices.append("android")
                elif plat in {"linux", "macos", "windows"}:
                    if plat not in file_flutter_devices:
                        file_flutter_devices.append(plat)
                elif plat in {"ios", "iphone", "ipad", "iphone simulator"}:
                    if "ios" not in file_flutter_devices:
                        file_flutter_devices.append("ios")
                elif plat in {"web", "web-server", "chrome", "edge", "firefox", "safari"}:
                    if "web" not in file_flutter_devices:
                        file_flutter_devices.append("web")
        if not file_flutter_devices and LINUX_HEADLESS_HINTS_RE.search(content_for_triggers):
            if "linux" not in file_flutter_devices:
                file_flutter_devices.append("linux")

    # anywhere fallbacks
    fb_trig_labels, fb_trig_groups = collect_hits_with_groups(
        TRIGGER_PATTERNS_ANYWHERE, content_for_triggers.lower()
    )
    if fb_trig_labels:
        trig_labels = unique_preserve(trig_labels + fb_trig_labels)
        trig_groups = unique_preserve(trig_groups + fb_trig_groups)

    # --- Guard 2: drop Third_Party_Lab when it's "upload-only" / non-test
    has_test_trigger = bool(trig_labels)  # includes Flutter/fallbacks if present
    if "Third_Party_Lab" in dev_groups:
        inline = detect_inline_env(content_for_triggers)
        cfgref = detect_config_env(content_for_triggers)
        has_ftl_instr = bool(FTL_HAS_INSTRUMENTATION.search(content_for_triggers))
        bs_test_hint = re.search(
            r'(?mi)\b(browserstack|bstack)\b[^\n]*\b(espresso|instrumentation|app-automate|automate|--device|--devices)\b',
            content_for_triggers
        )
        if (not has_test_trigger) and (not inline) and (not cfgref) and (not has_ftl_instr) and (not bs_test_hint):
            dev_labels = [l for l in dev_labels if l not in {
                "browserstack/bstack", "gcloud firebase", "saucectl",
                "appcenter test", "maestro cloud", "emulator.wtf action"
            }]
            if not any(l in {
                "browserstack/bstack", "gcloud firebase", "saucectl",
                "appcenter test", "maestro cloud", "emulator.wtf action"
            } for l in dev_labels):
                dev_groups = [g for g in dev_groups if g != "Third_Party_Lab"]

    # Guard for “other gha emulator” (skip if no Android context & no test trigger)
    android_context = bool(ANDROID_CONTEXT_RE.search(content_for_triggers))
    if ("other gha emulator" in dev_labels) and (not android_context) and (not has_test_trigger):
        dev_labels = [l for l in dev_labels if l != "other gha emulator"]
        if not dev_labels:
            dev_groups = [g for g in dev_groups if g != "Emulator"]

    # Android execution environment evidence for gating Flutter:
    ANDROID_SPECIFIC_3P_LABELS = {"gcloud firebase", "appcenter test", "emulator.wtf action", "saucectl"}
    has_android_env = (
        any(g in {"Emulator", "Real_Device"} for g in dev_groups) or
        any(lbl in ANDROID_SPECIFIC_3P_LABELS for lbl in dev_labels)
    )
    if flutter_it_hits and (flutter_it_android_targeted or has_android_env):
        file_flutter_it_androidish = True

    # normal device/gradle presence
    has_device_setup = bool(dev_labels)
    gradle_present   = bool(
        GRADLE_ANYWHERE_RE.search(content_for_triggers) or GRADLE_BUILD_ACTION_RE.search(content_for_triggers)
    )

    # accumulate
    file_device_labels  = unique_preserve(file_device_labels  + dev_labels)
    file_device_groups  = unique_preserve(file_device_groups  + dev_groups)
    file_trigger_labels = unique_preserve(file_trigger_labels + trig_labels)
    file_trigger_groups = unique_preserve(file_trigger_groups + trig_groups)
    file_has_test_trigger = file_has_test_trigger or bool(trig_labels)
    if has_device_setup and any(g in {"Emulator", "Third_Party_Lab", "Real_Device"} for g in dev_groups):
        file_has_device_with_group = True
    file_gradle_present_any = file_gradle_present_any or gradle_present

    # ---- Instrumentation CI signal (file-level, single "job")
    non_flutter_triggers = [l for l in file_trigger_labels if l.lower() != "flutter integration test"]
    flutter_counts_as_instr = (
        ("flutter integration test" in {l.lower() for l in file_trigger_labels}) and
        (file_flutter_it_androidish)
    )
    instru_t_ci_signal_file = bool(
        non_flutter_triggers
        or flutter_counts_as_instr
        or (file_has_device_with_group)
    )
    file_instru_signal_any = instru_t_ci_signal_file

    # Confidence & reasons (same scheme as YAML, but without AndroidTest boost)
    reasons = []
    if file_has_test_trigger:
        reasons.append("test_trigger: " + ", ".join(file_trigger_labels))
    if file_has_device_with_group:
        msg = (
            "device_setup: " + ", ".join(file_device_labels)
            if file_device_labels else "device_setup"
        )
        if file_gradle_present_any:
            msg += "; gradle present"
        reasons.append(msg)
    reason = " | ".join(r for r in reasons if r)

    confidence = ""
    if file_instru_signal_any and file_has_device_with_group and file_has_test_trigger:
        confidence = "high"
    elif file_instru_signal_any:
        confidence = "medium"

    # === map invocations & envs ===
    def has_gmd_gradle_trigger(trigger_labels: List[str]) -> bool:
        L = {l.lower() for l in trigger_labels}
        if any("manageddevice androidtest" in l for l in L):
            return True
        if ("variant/device androidtest" in L and not any(x in L for x in {
            "connected.*android.*", "connectedandroidtest", "connectedbenchmarkandroidtest",
            "spoon", "marathon"
        })):
            return True
        return False

    def has_connected_gradle_trigger(trigger_labels: List[str]) -> bool:
        L = {l.lower() for l in trigger_labels}
        keys = {
            "connected.*android.*", "connectedandroidtest", "connectedbenchmarkandroidtest",
            "connectedcheck", "cat shorthand", "connected (anywhere)",
            "connectedandroidtest (anywhere)", "connectedcheck (anywhere)",
            "spoon", "marathon", "devicecheck", "gha gradle inputs/script"
        }
        return any(k in L for k in keys) or any(
            ("connected" in l and "android" in l and "test" in l) for l in L
        )

    def has_baselineprofile_trigger(trigger_labels: List[str]) -> bool:
        L = {l.lower() for l in trigger_labels}
        return any("baselineprofile" in l for l in L)

    def map_test_invocations(groups: List[str], trigger_labels: List[str]) -> List[str]:
        s_groups = set(groups)
        L = {l.lower() for l in trigger_labels}
        out: List[str] = []
        if has_gmd_gradle_trigger(trigger_labels):
            out.append("Gradle_GMD")
        if has_connected_gradle_trigger(trigger_labels):
            out.append("Gradle_Connected")
        if has_baselineprofile_trigger(trigger_labels):
            out.append("Gradle")
        if "ADB" in s_groups:
            out.append("ADB")
        if "Third_Party_Lab" in s_groups:
            out.append("3P CLIs")

        # Flutter → add 3P CLIs only if Android evidence and no Gradle/ADB already
        has_gradle_or_adb = any(x in out for x in ["Gradle_GMD", "Gradle_Connected", "Gradle", "ADB"])
        if (not has_gradle_or_adb) and ("flutter integration test" in L) and file_flutter_it_androidish:
            out.append("3P CLIs")

        return sorted(set(out))

    def map_execution_envs(groups: List[str], labels: List[str]) -> List[str]:
        envs = set()
        s_groups, s_labels = set(groups), set(labels)
        if "Third_Party_Lab" in s_groups:
            envs.add("Third Party")
        if "Real_Device" in s_groups:
            envs.add("Real Device")
        if "Emulator" in s_groups:
            if "reactivecircus runner" in s_labels:
                envs.add("Emulator_ReactiveCircus")
            elif "malinskiy runner" in s_labels:
                envs.add("Emulator_Malinskiy")
            elif ("other gha emulator" in s_labels) or ("circleci android orb" in s_labels):
                envs.add("Emulator_Other")
            else:
                envs.add("Emulator_DIY")
        return sorted(envs)

    combined_groups = unique_preserve(file_trigger_groups + file_device_groups)
    test_inv = ",".join(map_test_invocations(combined_groups, file_trigger_labels))
    exec_envs = map_execution_envs(file_device_groups, file_device_labels)

    # 3P label detail
    third_party_label = ""
    if "Third Party" in exec_envs:
        if file_env_declared and file_env_location == "inline":
            third_party_label = "Third-Party Lab — Explicit Inline Env"
        elif file_env_declared and file_env_location == "config":
            third_party_label = "Third-Party Lab — Config-Referenced Env"
        else:
            third_party_label = "Third-Party Lab — Invocation Only"

    return {
        "instru_t_ci_signal": bool(file_instru_signal_any),
        "confidence": confidence if file_instru_signal_any else "",
        "confidence_reason": reason if file_instru_signal_any else "",
        "execution_environment": ",".join(exec_envs),
        "test_invocation": test_inv,
        "flutter_integ_t_signal": bool(file_flutter_it_present),
        "flutter_integ_t_d": ",".join(sorted(set(file_flutter_devices))),
        "third_party_env_label": third_party_label,
    }

# === Main: tie script index + YAML-call metadata together ===

def main():
    # 1) Index all script files in All_Config_Files
    script_index = build_script_index(CONFIG_DIR)

    # 2) Load metadata about which scripts are called from which YAML files
    meta_df = pd.read_csv(SCRIPTS_META_CSV)

    # Normalize key columns
    meta_df["full_name"] = meta_df["full_name"].astype(str).str.lower()
    meta_df["script_basename"] = meta_df["script_basename"].astype(str)
    meta_df["script_basename_lower"] = meta_df["script_basename"].str.lower()

    # Aggregate per (full_name, script_basename_lower) so that each physical script
    # file will have one row, with all callers merged.
    grouped = (
        meta_df
        .groupby(["full_name", "script_basename_lower"], as_index=False)
        .agg({
            "yml_filename": lambda s: ";".join(sorted({str(x) for x in s})),
            "script_path":  lambda s: ";".join(sorted({str(x) for x in s})),
        })
    )

    rows: List[Dict[str, Any]] = []

    for _, row in grouped.iterrows():
        full_name = row["full_name"]
        script_base = row["script_basename_lower"]
        key = (full_name, script_base)

        matches = script_index.get(key, [])
        if not matches:
            # Nothing found in All_Config_Files for this script; skip but warn.
            print(f"[WARN] No script file found in All_Config_Files for: {full_name} / {script_base}")
            continue

        called_from_ymls = row["yml_filename"]
        script_paths_meta = row["script_path"]

        for path, ci_platform in matches:
            info = analyze_script_file(path)
            csv_row = {
                "filename": path.name,
                "full_name": full_name,
                "ci_platform": ci_platform,
                "instru_t_ci_signal": info["instru_t_ci_signal"],
                "confidence": info["confidence"],
                "confidence_reason": info["confidence_reason"],
                "execution_environment": info["execution_environment"],
                "test_invocation": info["test_invocation"],
                "flutter_integ_t_signal": info["flutter_integ_t_signal"],
                "flutter_integ_t_d": info["flutter_integ_t_d"],
                "third_party_env_label": info["third_party_env_label"],
                "called_from_ymls": called_from_ymls,
                "script_paths": script_paths_meta,
            }
            rows.append(csv_row)

    out_df = pd.DataFrame(rows)
    out_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"Saved: {OUTPUT_CSV} (rows={len(out_df)})")

if __name__ == "__main__":
    main()


[WARN] No script file found in All_Config_Files for: 2468785842.krkr2 / dotnet-install.sh
[WARN] No script file found in All_Config_Files for: activitywatch.aw-android / install-ndk.sh
[WARN] No script file found in All_Config_Files for: alibaba.mnn / install.sh
[WARN] No script file found in All_Config_Files for: android.codelab-android-workmanager / test_all_ftl.sh
[WARN] No script file found in All_Config_Files for: ankidroid.anki-android / update-js-libs.sh
[WARN] No script file found in All_Config_Files for: appdev.siyuan-unlock / get-tdm-gcc.sh
[WARN] No script file found in All_Config_Files for: arakiken.mlterm / autogen.sh
[WARN] No script file found in All_Config_Files for: atsign-foundation.noports / entrypoint.sh
[WARN] No script file found in All_Config_Files for: audio4linux.jdsp4linux / qt512-env.sh
[WARN] No script file found in All_Config_Files for: audio4linux.jdsp4linux / qt515-env.sh
[WARN] No script file found in All_Config_Files for: audio4linux.jdsp4linux / qt56-e